<img src=images/gdd-logo.png width=300px align=right>

# Classification

In this notebook, you will classify penguins species based on bodily measurements using the Scikit-Learn API. 

You will first be introduced to the Penguins dataset and the Scikit-Learn library. Afterwards, you will learn the following aspects:

- [Loading in the data](#loading-in-the-data)    
    - [<mark>Exploring the dataset</mark>](#exploring-the-dataset) 
    - [Visualising the dataset](#visualising-the-dataset)  
- [Preparing the data for sklearn](#preparing)
    - [Splitting the dataset](#train-test-split)
- [Model creation & evaluation](#model)
    - [Training and evaluating a Scikit-Learn model](#steps)
    - [Model Visualisation](#vis)
    - [<mark>Choosing a different model</mark>](#choosing-models)
    - [Alternative metrics](#metrics)
    - [<mark>Precision and recall with Scikit-Learn</mark>](#precision-recall)
    - [Prediction and Inference](#inference)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

---
## About the data
The data was collected and made available by Dr. Kristen Gorman and the Palmer Station, Antartica LTER. Their goal was to provide a great dataset for data exploration, visualisation and - in this case - a demonstration of the Scikit-Learn API. 

The data set contains measurements for different species of penguins living at the Palmer station:

|Field|Description|
|:---|:---|
|species|The species of the penguin: Adelie, Chinstrap or Gentoo|
|island|The island on which the penguin was spotted|
|bill_length_mm|The length of the penguin's bill in mm|
|bill_depth_mm|The depth of the penguin's bill in mm|
|flipper_length_mm|The length of the penguin's flipper in mm|
|body_mass_g|The weight of the penguin in grams|
|sex|The gender of the penguin - Female or Male|


<img src="images/02_Classification_Penguins/palmer-penguins.png" width="500" align='left'><img src="images/02_Classification_Penguins/culmen_depth.png" width="400" align='right'>

---
## Scikit-Learn
Scikit-Learn is *the* library for machine learning in Python. You could consider it the swiss army knife of machine learning. A wide variety of machine learning models are implemented by the community and core developers, with a consistent API. Once you master this API, it's easy to apply a wide variety of machine learning algorithms, and you have a handy tool to help you out with preprocessing, model evaluation and model selection. 

#### Why Scikit-Learn?
- Many available machine learning models
- Models are implemented by an expert team and checked by a large community
- Covers most machine-learning tasks
- Commitment to documentation, consistency and usability
- Designed to work with other key Python libraries (NumPy, Pandas etc)

<a id = 'loading-in-the-data'></a>
## 1. Loading in the data

There are many places your data can originate from. Maybe you want to load it from a Excel file you have stored locally on your system, maybe you have a .csv file stored online somewhere. Scikit-learn comes with various standard datasets that can be used for practice, that can be loaded if you have Scikit-Learn installed on your system. 

Our dataset will be loaded in as a Pandas dataframe and can be used as such. Pandas is a powerful library for data wrangling.

In [ ]:
penguins = pd.read_csv('data/penguins.csv')
penguins.head(10)

<a id = 'exploring-the-dataset'></a>
## <mark> Exercise: Exploring the dataset </mark>

Below are some typical things you may want to check as part of your initial investigation of the dataset.

1. How many rows and columns are present in the data?

2. Which data types are used by each column?

3. Are there any missing values?

4. How many species are there?

5. How many penguins are there for each species?

In [ ]:
# %load answers/02_Classification_Penguins/exploration.py

<a id = 'visualising-the-dataset'></a>
## Visualising the dataset 

To understand the dataset better it can be useful to create some visualisations.

Below is a  histogram of the penguin's flipper lengths:

In [ ]:
sns.histplot(data=penguins, x='flipper_length_mm')

You can use visualisations to examine how different the data is for the different species.

For example, here is a histogram of flipper lengths *for the different species*. Would you be able to separate the species based on this measurement alone?

In [ ]:
sns.histplot(data=penguins, x='flipper_length_mm', hue='species')

Let's examine the relationship between two variables.

Below is a scatter plot of flipper length vs. body mass:

In [ ]:
sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g')

It may be easier to distinguish different species when looking at more than one variable.

Here is a a scatter plot of flipper length vs. body mass *for the different species*. Would you be able to separate the species based on the relationship between these measurements?

In [ ]:
sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g', hue='species')

Seaborn also allows us to see this information for each numeric variable:

In [ ]:
sns.pairplot(data=penguins, hue='species')

<a id = 'preparing'></a>
## 2. Preparing the data for Scikit-Learn

The first thing you might notice here is that there are some data point entries that have no value - the value simply says `NaN`. This means this information is missing. 

In [ ]:
(
    penguins
    .loc[penguins.isnull().any(axis=1)]
)

Unfortunately, that also means the information cannot be used as is to create a machine learning model with Scikit-Learn. You must find a way to deal with the missing values. 

There are multiple strategies for dealing with missing data. For example, you could replace a missing values with the mean of the column. For example, if for a particular penguin the value for body mass is missing, **you could replace the `NaN` with the mean** recorded body mass of all penguins. 

Scikit-Learn even provides us with a great interface to apply such transformations, but more about this later. 

For the moment, however, let's simply discard all the incomplete data points with pandas `.dropna()` functionality. 

In [ ]:
penguins_cleaned = penguins.dropna()
penguins_cleaned.head()

Second of all, you might have noticed that your data contains more information than only the measurements `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` and `body_mass_g`.

The data also contains the `sex` of the penguin and the `island` where the penguin was spotted. You could incorporate this extra information. However, this requires some extra preprocessing outside of the scope of this notebook. So for simplicity, let's  focus on the four measurements first.

🔔 Let's use the power of Pandas to create a **feature matrix** $X$ and **target vector** $y$.

In [ ]:
feature_columns = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

X = penguins_cleaned.loc[:, feature_columns]
y = penguins_cleaned.loc[:, 'species']

print(f'The shape of feature matrix X is: {X.shape}')
print(f'The shape of target vector y is: {y.shape}')

The target vector $y$ is the output of the model. It is also known as the dependent variable; what the model is trying to predict.

The feature matrix $X$ contains the input of the model. This is also known as the independent or predictive variables:  the variables that contain the information that the model is going to use to generate predictions.


A feature matrix $X$ consists of $n$ samples with $m$ features - in this case $n=333$ and $m=4$.

In [ ]:
X.head()

Each row in the feature matrix $X$ corresponds to a value in the target vector $y$.

In [ ]:
y.head()

In [ ]:
y.unique()

Our model will then attempt to learn a relationship that can map a row in $X$ to the corresponding value in $y$.

<a id = 'train-test-split'></a>
### Splitting the dataset
🔔 An important goal of machine learning is to create a model that does not only do well on the data that it has already seen, but **will also perform well under new circumstances** on data that it has not seen before. In machine learning, this concept is called _**generalization**_. 

Imagine this: 
> Penguin *Lucy* (with: `bill_length_mm=33`, `bill_depth_mm=16`, `flipper_length_mm=180`, and `body_mass_g=3500`) is a Gentoo. 

<img src="images/02_Classification_Penguins/gentoo.jpg" width="300">
<center><font size=1>Picture of Penguin <i>Lucy</i><font></center>

*Penguin Lucy* was presented during the training of our model. That means, it was one of the examples that the algorithm used to create an understanding of what a Gentoo penguin looks like and how you can distinguish it from a Chinstrap or Adélie. 

> If you want to know how well your model does, asking the model to classify the same penguin it was trained on does not give you a lot of information on the model's performance. 

<mark>**Question:**</mark> **Why does it not tell us much about the model's performance?**

<br>  
<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
The problem is that you don't know whether it has truly learned the relationship between the features and the targets, or whether has it simply memorized the data is has already seen.
    
</details>    


🔔 That's why it is common to separate your dataset into two parts:
* The **_training_ set**: This is the data (features and targets) that will guide the learning process. 
* The **_test_ set**: This is the data (features and targets) used to _evaluate_ how well the model has learned the relationship between features and target. 

<img src="images/02_Classification_Penguins/train-test.png" width="400">

Scikit-Learn's `train_test_split` function allows us to split the data in a train- and test set. By default, the test set size is set to 25% and the data is shuffled. 

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_test_split?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

print(f'The size of our feature matrix for the train set is: {X_train.shape}')
print(f'The size of our target vector for the train set is: {y_train.shape}')

print(f'\nThe size of our feature matrix for the test set is: {X_test.shape}')
print(f'The size of our target vector for the test set is: {y_test.shape}')

Let's see if our data is in fact shuffled: 

In [ ]:
y_test.values[:25]

<a id = 'model'></a>
## 3. Model creation and evaluation

Now you are ready to create our machine learning model! 

Scikit-Learn has a rich collection of algorithms readily available. Depending on the case you are working on, Scikit-Learn most likely has a model that will suit your purposes. 

<a id = 'steps'></a>
## Training a Scikit-Learn model

Below are the steps for training a model using the Scikit-Learn API 
1. Choose a model class and import it.
2. Choose the model *hyperparameters* by instantiating the model class with desired values.
    - Hyperparameters are pre-specified values that determine the behaviour of a model before training. E.g. the maximun allowed depth of a decision tree, the number of trees on a random forest.
4. Train the model to the preprocessed train data by calling the `.fit()` method of the model instance.
5. Evaluate model's performance using available metrics.

In [ ]:
# Step 1: import the chosen algorithm 
from sklearn.tree import DecisionTreeClassifier

In [ ]:
DecisionTreeClassifier?

<img src="images/02_Classification_Penguins/tree.png" width="600">

In [ ]:
# Step 2: instantiate the model with the chosen hyperparameters
model = DecisionTreeClassifier(max_depth=2)

In [ ]:
# Step 3: train the model with the training data
model.fit(X_train, y_train)

You have now trained a model that can be used to make predictions on new data! 

Remember the test set? 

In [ ]:
X_test.head(10)

The test set is new, unseen data to the model that you can now create predictions on. 

In [ ]:
y_pred = model.predict(X_test)
y_pred[0:10]

You can now compare the predicted species against the true species to see how well your model does. 

In [ ]:
y_test[0:10].values

Fortunately, you don't have to do that comparison yourself. Scikit-Learn has made many implementations of possible metrics readily available, such as accuracy. 

🔔 $\text{accuracy} = \frac{correct}{total}$

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

Pretty good! 

Alternatively, you can use the `.score()` method. On a Decision Tree this will (by default) return the accuracy score:

In [ ]:
model.score(X_test, y_test)

This classification model can also return probabilities indicating how confidence it is about its predictions.

In [ ]:
pd.DataFrame(
    columns = y.unique(),
    data = model.predict_proba(X_test))

<a id = 'vis'></a>
## Model Visualisation

Another advantage of decision trees over some of the other available models is that decision trees are relatively easy to interpret. By visualising the tree-like structure of the decision tree, you can understand why the model classifies samples the way it does.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(14,10))

plot_tree(model, 
          ax=ax, 
          feature_names = feature_columns, 
          class_names = list(y.unique()));

<a id = 'choosing-models'></a>
## <mark>Exercise: Choosing a different model </mark>

What if you were interested in a model other than a Decision Tree? 

That's actually really easy! You simply replace the chosen model with another and the rest of the code can stay the same.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Uncomment the model that you want to try
model = DecisionTreeClassifier()
# model = RandomForestClassifier()
# model = KNeighborsClassifier()
# model = SVC()

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)
print(f'Model accuracy: {model.score(X_test, y_test)}')

<a id = 'metrics'></a>
## Alternative metrics

🔔 But accuracy is not the only metric you could be interested in. Alternatives are, for example, **precision** and **recall**. 

In [ ]:
# Re-train our decision tree
model = DecisionTreeClassifier(max_depth=2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

<!-- * _Precision_ is the proportion of positive identifications that was actually correct. 
* _Recall_ is the proportion of actual positives that was identified correctly.
* _F1 score_ is a function of precision and recall, that you use when you seek a balance between precision and recall.  -->

#### Precision & Recall

Predictions about a class fall into four categories:
* True Positive: Correctly predict item is that class
* True Negative: Correctly predict item is NOT that class
* False Positive: Incorrectly predict item is that class
* False Negative: Incorrectly predict item is NOT that class


<img src="images/02_Classification_Penguins/TPFN.png" width="350">

In a classification task, the **precision** for a class is the number of true positives (i.e. the number of items correctly labelled as belonging to the positive class) divided by the total number of elements labelled as belonging to the positive class (i.e. the sum of true positives and false positives, which are items incorrectly labelled as belonging to the class).

<img src="images/02_Classification_Penguins/precision.png" width="200">

As an example, the code below shows every time we predicted a penguin in the test set would be an Adelie.

In [ ]:
(
    X_test
    .assign(prediction = y_pred,
            true = y_test)
    .loc[lambda df: df['prediction'] == "Adelie"]
)

Precision indicates how often our Adelie predictions were correct.

In [ ]:
(
    X_test
    .assign(prediction = y_pred,
            true = y_test)
    .loc[lambda df: df['prediction'] == "Adelie"]
    .assign(correct_prediction = lambda df: df['prediction']==df['true'])
    ['correct_prediction'].mean()    
)

**Recall** in this context is defined as the number of true positives divided by the total number of elements that actually belong to the positive class (i.e. the sum of true positives and false negatives, which are items which were not labelled as belonging to the positive class but should have been).

<img src="images/02_Classification_Penguins/recall.png" width="200">

As an example, the code below shows our predictions for all the Adelie penguins in the test set.

In [ ]:
(
    X_test
    .assign(prediction = y_pred,
            true = y_test)
    .loc[lambda df: df['true'] == "Adelie"]
)

Recall indicates how often we correctly predicted an Adelie penguin.

In [ ]:
(
    X_test
    .assign(prediction = y_pred,
            true = y_test)
    .loc[lambda df: df['true'] == "Adelie"]
    .assign(correct_prediction = lambda df: df['prediction']==df['true'])
    ['correct_prediction'].mean()    
)

The differences between these metrics can be explained with this example:
Let's say you create a model that should classify email messages as spam or not spam. _Precision_ measures the percentage of emails **flagged** as spam that were correctly classified, while _recall_ measures the percentage of **actual** spam emails that were correctly classified. 

In certain cases, _precision_ can be important. For example, imagine you are training a model to detect fraudulent customer behavior. Of course, it is crucial to identify cases where customers are effectively stealing money, but wrongly accusing customers of fraud could result in high customer churn.

In a medical context, _recall_ (also called _sensitivity_) is often very important. If you mistakenly tell a person with a life-threatening disease that they're healthy, the consequences could be more severe than the other way around. 

<a id = 'precision-recall'></a>
## <mark>Exercise: Precision and Recall with Scikit-Learn </mark>

Earlier, you used the `accuracy_score` function from Scikit-Learn. However, Scikit-Learn has many more evalation metrics already built-in. Visit the documentation and calculate the precision (using [`precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score)) and recall (using [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html#sklearn.metrics.recall_score)) of your predictions.

**Note:** When you do not set the `average` parameter, it will default to `'binary'`, which will give an error in this case. Instead, set it to `None`.

<details>

  <summary><span style="color:blue">Show hint</span></summary>

Don't forget to import the functions:
    
```python
from sklearn.metrics import ...
```
    
</details>

In [ ]:
from sklearn.metrics import precision_score, recall_score
# add your code

What does the output represent? And why did the `'binary'` value of `average` give an error?

<details>

  <summary><span style="color:blue">Show answer</span></summary>

Because the data does not only have two classes ('binary' meaning 'two'), but actually three classes (species of penguins).
    
</details>

In [ ]:
# %load answers/02_Classification_Penguins/recall-precision.py

### Confusion Matrix and Classification Report

If the number of classes is not too large, it is also common to produce a confusion matrix to interpret how good the predicitions were.

The raw **confusion matrix** can be quickly acquired as shown below: 

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

Or better, you can use the `ConfusionMatrixDisplay` function to plot the confusion matrix:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=model.classes_)
disp.plot()

In `sklearn`, the classification report can give us a breakdown of the precision and recall for each species of penguin:

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred)
print(report)

Another common metric is the **F1 score**. It combines both precision and recall by calculating their harmonic mean:
<br><br>


$${\displaystyle F_{1}={\frac {2}{\mathrm {recall} ^{-1}+\mathrm {precision} ^{-1}}}=2\cdot {\frac {\mathrm {precision} \cdot \mathrm {recall} }{\mathrm {precision} +\mathrm {recall} }}={\frac {\mathrm {tp} }{\mathrm {tp} +{\frac {1}{2}}(\mathrm {fp} +\mathrm {fn} )}}}$$

<br>Precision, recall and F1 score (and many more) are all readily available [Scikit-Learn Metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics).

<a id='inference'></a>
## Prediction vs. Inference

🔔 Sometimes your goal may be ***inference*** rather than ***prediction***.

> **Prediction**: To use a model to *predict* new outcomes as best as possible.

> **Inference**: To use a model to *infer* the underlying data generating process. For example, which features had the most influence on our predictions?

Luckily, the `DecisionTreeClassifier` is well-suited for inference. You can simply find out which features were most important by looking at the `.feature_importances_` attribute.

In [ ]:
model.feature_importances_

In [ ]:
inference_df = pd.DataFrame(columns  = X.columns, data = [model.feature_importances_])
inference_df

---
# Summary

Scikit-Learn is an excellent, resourceful tool for machine learning in Python. Now, you should know how you can split a dataset with `train_test_split` into a training and test set, how to create and train a model, how to use the trained model to create predictions, and how to use the tools from `sklearn.metrics` to evaluate how good the model is. 

<img src="images/02_Classification_Penguins/penguin-bye.png" width="300">

*Image generated by DALL-E*